# Crypto.Net · 03 — Ethereum and EVM

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./03-ethereum-evm.id.ipynb)

EIP-55 addresses, live reads from Sepolia, an ERC-20 contract, EIP-1559 transactions, EIP-191 and EIP-712 signatures, and the ABI codec.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using System.Numerics;
using Crypto.Net.Core;
using Crypto.Net.Wallet;
using Crypto.Net.Evm;

var wallet  = HdWallet.Generate();
var account = wallet.GetEvmAccount(0, EvmChain.Sepolia);   // m/44'/60'/0'/0/0, like MetaMask
account.Address.Value

## Addresses (EIP-55)

Mixed-case addresses carry a checksum; a single wrong letter is caught.

In [ ]:
string good = "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAed";
string bad  = "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAeD";
Console.WriteLine($"{good} valid: {EvmAddress.IsValid(good)}");
Console.WriteLine($"{bad} valid: {EvmAddress.IsValid(bad)}");
Console.WriteLine($"checksummed: {EvmAddress.ToChecksumAddress(good.ToLowerInvariant())}");

## Live data from Sepolia

In [ ]:
var client = new EvmRpcClient(EvmChain.Sepolia);
Console.WriteLine($"chain id {await client.GetChainIdAsync()}, block {await client.GetBlockNumberAsync():N0}");
var fees = await client.EstimateFeesAsync();
Console.WriteLine($"base fee {Amount.FromWei(fees.BaseFee).WithDecimals(18)} ETH, suggested max {Amount.FromWei(fees.MaxFeePerGas)} ETH per gas");
Console.WriteLine($"your balance: {await client.GetBalanceAsync(account.Address)} ETH");

## ERC-20

`Erc20Client` reads metadata and scales balances by `decimals()`. This is Sepolia WETH.

In [ ]:
var weth = new Erc20Client(client, "0xfFf9976782d46CC05630D1f6eBAb18b2324d6B14");
Console.WriteLine($"{await weth.GetNameAsync()} ({await weth.GetSymbolAsync()}), {await weth.GetDecimalsAsync()} decimals");
Console.WriteLine($"total supply {await weth.GetTotalSupplyAsync()}");

## EIP-1559 transaction

Built and signed offline, then decoded again to recover the sender.

In [ ]:
var tx = new EvmTransaction
{
    ChainId = EvmChain.Sepolia.NumericChainId,
    Nonce = 0,
    MaxPriorityFeePerGas = 1_000_000_000,
    MaxFeePerGas = 20_000_000_000,
    GasLimit = 21_000,
    To = "0xfFf9976782d46CC05630D1f6eBAb18b2324d6B14",
    Value = Amount.FromEther(0.001m).BaseUnits,
};
var signed  = account.SignTransaction(tx);
var decoded = EvmTransaction.DecodeSigned(signed.RawBytes);

Console.WriteLine($"hash   {signed.Hash}");
Console.WriteLine($"raw    {signed.RawHex}");
Console.WriteLine($"type   {decoded.Transaction.Type}, nonce {decoded.Transaction.Nonce}, value {Amount.FromWei(decoded.Transaction.Value)} ETH");
Console.WriteLine($"sender {decoded.RecoverSender()} (ours: {decoded.RecoverSender() == account.Address.Value})");

## Message signatures

EIP-191 `personal_sign` ("Sign in with Ethereum") and EIP-712 typed data.

In [ ]:
byte[] sig = account.SignMessage("Sign in to Crypto.Net");
Console.WriteLine($"signature {HexUtil.Encode(sig)}");
Console.WriteLine($"recovered {EvmMessageSigner.RecoverPersonalMessageSigner("Sign in to Crypto.Net", HexUtil.Encode(sig))}");

string typedData = """
{
  "types": {
    "EIP712Domain": [ {"name":"name","type":"string"}, {"name":"version","type":"string"}, {"name":"chainId","type":"uint256"} ],
    "Order": [ {"name":"maker","type":"address"}, {"name":"amount","type":"uint256"}, {"name":"note","type":"string"} ]
  },
  "primaryType": "Order",
  "domain": { "name": "Crypto.Net Demo", "version": "1", "chainId": 11155111 },
  "message": { "maker": "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAed", "amount": "1000000", "note": "hello" }
}
""";
byte[] typedSig = account.SignTypedData(typedData);
Console.WriteLine($"EIP-712 digest {HexUtil.Encode(EvmMessageSigner.HashTypedData(typedData))}");
Console.WriteLine($"EIP-712 signer {EvmMessageSigner.RecoverTypedDataSigner(typedData, typedSig)}");

## ABI

Encode calls by signature, decode return data and logs.

In [ ]:
byte[] call = EvmAbi.EncodeFunctionCall("transfer(address to, uint256 amount)", account.Address.Value, new BigInteger(1_000_000));
Console.WriteLine(HexUtil.Encode(call));

byte[] encoded = EvmAbi.Encode("string,uint256[],(address,bool)", "Crypto.Net", new object[] { 1, 2, 3 }, new object[] { account.Address.Value, true });
var values = EvmAbi.Decode("string,uint256[],(address,bool)", encoded);
Console.WriteLine($"{values[0]} · [{string.Join(", ", (object?[])values[1]!)}] · {string.Join(", ", (object?[])values[2]!)}");
Console.WriteLine($"Transfer topic {Erc20Client.TransferEventTopic}");

## Sending for real

With Sepolia ETH in the account:

```csharp
await using var signer = account.CreateSigner();
TxHash hash = await client.TransferAsync(signer, "0x…", Amount.FromEther(0.001m)); // nonce, fees, gas, simulation
TxReceipt receipt = await client.WaitForReceiptAsync(hash);
```

In [ ]:
account.Dispose(); client.Dispose(); wallet.Dispose();
"disposed"